# BBO Capstone — Round 8 Query Generation
**Imperial Business School | Executive Master in ML/AI**  
**Candidate:** Gian Franco Cattaneo  
**Module:** 19.1 — Refining Strategies for Black-Box Optimisation  
**Date:** 2026-05-18  

---

## Pipeline Architecture
- **GP Kernel:** Matérn-5/2 (ARD) + WhiteKernel (noise) + ConstantKernel (amplitude)
- **Scaler:** StandardScaler on inputs (zero-mean, unit-variance)
- **Acquisition:** Expected Improvement (EI) — maximisation formulation
- **Optimiser:** L-BFGS-B with 35 random restarts
- **Objective:** All 8 functions treated as maximisation targets
- **Data:** 7 rounds × 8 functions = 56 evaluated query points

---

In [1]:
# ============================================================
# CELL 1 — IMPORTS AND GLOBAL CONFIGURATION
# ============================================================
import numpy as np
import warnings
from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

# Global optimiser settings
N_RESTARTS    = 35      # L-BFGS-B restarts for acquisition optimisation
N_CANDIDATES  = 5000    # Random candidates for acquisition warm-start
DOMAIN_LOWER  = 0.0
DOMAIN_UPPER  = 0.999999

print('Libraries loaded successfully.')
print(f'  GP restarts       : {N_RESTARTS}')
print(f'  Acquisition cands : {N_CANDIDATES}')
print(f'  Domain            : [{DOMAIN_LOWER}, {DOMAIN_UPPER}]^d')

Libraries loaded successfully.
  GP restarts       : 35
  Acquisition cands : 5000
  Domain            : [0.0, 0.999999]^d


In [2]:
# ============================================================
# CELL 2 — COMPLETE DATASET: ROUNDS 1–7
# ============================================================
# Each list entry is one round of queries.
# Index 0 = f1 (d=2), Index 7 = f8 (d=8).
# All functions: maximisation objective.

all_inputs = [
    # --- Round 1 (W12, 26/04/26) ---
    [np.array([0.034388, 0.909319]),
     np.array([0.695196, 0.395970]),
     np.array([0.548145, 0.174647, 0.303245]),
     np.array([0.440429, 0.425456, 0.378357, 0.397088]),
     np.array([0.000000, 0.675974, 0.999999, 0.999999]),
     np.array([0.464677, 0.242110, 0.574863, 0.999999, 0.000000]),
     np.array([0.000000, 0.241713, 0.327655, 0.218095, 0.375335, 0.747501]),
     np.array([0.064016, 0.008062, 0.123268, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])],

    # --- Round 2 (W13, 06/04/26) ---
    [np.array([0.999999, 0.999999]),
     np.array([0.698486, 0.000000]),
     np.array([0.850892, 0.035316, 0.936193]),
     np.array([0.999999, 0.000000, 0.000000, 0.365908]),
     np.array([0.000000, 0.000000, 0.999999, 0.999999]),
     np.array([0.142734, 0.321812, 0.416483, 0.999999, 0.304415]),
     np.array([0.000000, 0.302741, 0.000000, 0.187177, 0.000000, 0.167183]),
     np.array([0.096074, 0.000000, 0.581701, 0.000000, 0.999999, 0.383890, 0.202188, 0.999999])],

    # --- Round 3 (W14, 08/04/26) ---
    [np.array([0.250000, 0.250000]),
     np.array([0.695000, 0.396000]),
     np.array([0.300000, 0.500000, 0.700000]),
     np.array([0.440000, 0.425000, 0.378000, 0.397000]),
     np.array([0.000000, 0.850000, 0.999999, 0.999999]),
     np.array([0.500000, 0.500000, 0.500000, 0.500000, 0.500000]),
     np.array([0.000000, 0.242000, 0.328000, 0.218000, 0.375000, 0.748000]),
     np.array([0.064000, 0.008000, 0.120000, 0.000000, 0.999999, 0.382000, 0.031000, 0.806000])],

    # --- Round 4 (W15, 19/04/26) ---
    [np.array([0.500000, 0.500000]),
     np.array([0.700000, 0.200000]),
     np.array([0.950000, 0.010000, 0.990000]),
     np.array([0.999999, 0.000000, 0.000000, 0.700000]),
     np.array([0.000000, 0.000000, 0.500000, 0.500000]),
     np.array([0.300000, 0.400000, 0.600000, 0.200000, 0.600000]),
     np.array([0.000000, 0.150000, 0.000000, 0.100000, 0.000000, 0.100000]),
     np.array([0.100000, 0.000000, 0.800000, 0.000000, 0.999999, 0.380000, 0.350000, 0.999999])],

    # --- Round 5 (W16, 23/04/26) ---
    [np.array([0.472781, 0.505546]),
     np.array([0.695211, 0.395970]),
     np.array([0.511275, 0.215264, 0.371049]),
     np.array([0.455000, 0.415000, 0.385000, 0.395000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.758817, 0.272673, 0.522143, 0.999999, 0.000000]),
     np.array([0.000000, 0.260000, 0.340000, 0.232000, 0.395000, 0.752000]),
     np.array([0.040000, 0.000000, 0.090000, 0.005000, 0.999999, 0.367013, 0.020000, 0.780000])],

    # --- Round 6 (W17, 01/05/26) ---
    [np.array([0.445562, 0.511092]),
     np.array([0.693000, 0.397000]),
     np.array([0.490000, 0.230000, 0.395000]),
     np.array([0.430000, 0.430000, 0.375000, 0.400000]),
     np.array([0.005000, 0.999999, 0.999999, 0.999999]),
     np.array([0.450000, 0.240000, 0.580000, 0.999999, 0.000000]),
     np.array([0.000000, 0.238000, 0.325000, 0.215000, 0.370000, 0.743000]),
     np.array([0.063000, 0.008000, 0.123000, 0.000000, 0.999999, 0.382000, 0.031000, 0.807000])],

    # --- Round 7 (W18, 06/05/26) ---
    [np.array([0.475000, 0.503000]),
     np.array([0.697000, 0.393000]),
     np.array([0.478000, 0.223000, 0.408000]),
     np.array([0.420000, 0.440000, 0.373000, 0.403000]),
     np.array([0.000000, 0.999999, 0.999999, 0.999999]),
     np.array([0.468000, 0.241000, 0.572000, 0.999999, 0.000000]),
     np.array([0.000000, 0.235000, 0.322000, 0.212000, 0.367000, 0.740000]),
     np.array([0.064016, 0.008062, 0.124000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])]
]

all_outputs = [
    # Round 1
    [-2.4674747069022486e-270, 0.7237404632835625, -0.08911956876452833,
      0.25957575200735095, 2105.928152398213, -0.5507747202906804,
      2.207308607344047, 9.8595486103895],
    # Round 2
    [1.517648729565899e-192, 0.5297658866453171, -0.23982430098711077,
     -27.859767965401783, 1616.625747348229, -1.0045153236844038,
      0.050978228653516464, 9.2933769573024],
    # Round 3
    [9.797748409814019e-42, 0.5263661301012157, -0.1139602029925284,
      0.2748080020297299, 2932.694991178572, -1.0159268487405835,
      2.2071746109147172, 9.8591545999995],
    # Round 4
    [2.6752879910742468e-09, 0.5813540452269076, -0.4594065810473597,
     -30.894440825162423, 83.9625, -1.223884840915805,
      0.02363347322274405, 8.5129002799994],
    # Round 5
    [8.168635327996585e-08, 0.6238852457166373, -0.0707083820875107,
     -0.3996600230633507, 4440.480873479282, -0.9105784720492842,
      2.113257173327904, 9.8387496578305],
    # Round 6
    [-5.316626716773722e-07, 0.3979411317837673, -0.05294904589920826,
      0.4636173326649424, 4440.482959868813, -0.5765502837220897,
      2.2377743369228718, 9.8591202999995],
    # Round 7
    [1.3110732833867364e-07, 0.4872120229835646, -0.03529824725945802,
      0.36348229336051263, 4440.480873479282, -0.636071744816847,
      2.250193135816396, 9.8595765698615]
]

# Reorganise: per-function lists of (X, y)
N_FUNCTIONS = 8
N_ROUNDS    = len(all_inputs)

func_X = [[] for _ in range(N_FUNCTIONS)]
func_y = [[] for _ in range(N_FUNCTIONS)]

for rnd in range(N_ROUNDS):
    for fi in range(N_FUNCTIONS):
        func_X[fi].append(all_inputs[rnd][fi])
        func_y[fi].append(all_outputs[rnd][fi])

# Convert to numpy arrays
for fi in range(N_FUNCTIONS):
    func_X[fi] = np.vstack(func_X[fi])
    func_y[fi] = np.array(func_y[fi])

dims = [func_X[fi].shape[1] for fi in range(N_FUNCTIONS)]

print('Dataset loaded: 7 rounds × 8 functions')
print(f'{"Func":>6} {"Dims":>6} {"N pts":>6} {"y_best":>14} {"y_mean":>14}')
print('-' * 52)
for fi in range(N_FUNCTIONS):
    print(f'  f{fi+1:>2}  {dims[fi]:>5}  {len(func_y[fi]):>5}  '
          f'{func_y[fi].max():>14.6f}  {func_y[fi].mean():>14.6f}')

Dataset loaded: 7 rounds × 8 functions
  Func   Dims  N pts         y_best         y_mean
----------------------------------------------------
  f 1      2      7        0.000000       -0.000000
  f 2      2      7        0.723740        0.552895
  f 3      3      7       -0.035298       -0.151609
  f 4      4      7        0.463617       -8.256055
  f 5      4      7     4440.482960     2865.808014
  f 6      5      7       -0.550775       -0.845472
  f 7      6      7        2.250193        1.584331
  f 8      8      7        9.859577        9.583204


In [3]:
# ============================================================
# CELL 3 — GP-BO PIPELINE
# ============================================================

def build_kernel(d):
    """
    Construct the GP kernel:
      ConstantKernel (amplitude) × Matern-5/2 (ARD, one length-scale per dim)
      + WhiteKernel (homoscedastic noise)
    ARD is enabled by passing length_scale as a d-vector.
    """
    return (ConstantKernel(1.0, (1e-3, 1e3))
            * Matern(length_scale=np.ones(d),
                     length_scale_bounds=(1e-3, 10.0),
                     nu=2.5)
            + WhiteKernel(noise_level=1e-4,
                          noise_level_bounds=(1e-8, 1e-1)))


def fit_gp(X_raw, y):
    """
    Scale inputs, fit GP, return (gp, scaler).
    Uses 10 L-BFGS-B restarts for kernel hyperparameter optimisation.
    """
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_raw)
    d      = X_raw.shape[1]
    gp = GaussianProcessRegressor(
        kernel=build_kernel(d),
        n_restarts_optimizer=10,
        normalize_y=True,
        random_state=42
    )
    gp.fit(X_sc, y)
    return gp, scaler


def expected_improvement(x, gp, scaler, y_best, xi=0.01):
    """
    Expected Improvement for MAXIMISATION.

    EI(x) = (μ(x) - y_best - ξ) · Φ(Z) + σ(x) · φ(Z)
    Z      = (μ(x) - y_best - ξ) / σ(x)

    Returns negative EI (minimisation target for scipy.optimize).
    """
    x_sc       = scaler.transform(x.reshape(1, -1))
    mu, sigma  = gp.predict(x_sc, return_std=True)
    mu, sigma  = mu[0], max(sigma[0], 1e-9)
    improvement = mu - y_best - xi
    Z           = improvement / sigma
    ei          = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -max(ei, 0.0)   # negative for minimiser


def optimise_acquisition(gp, scaler, y_best, d, n_restarts=N_RESTARTS,
                          n_candidates=N_CANDIDATES):
    """
    Maximise EI over [0, 0.999999]^d via multi-start L-BFGS-B.

    Strategy:
      1. Evaluate EI at n_candidates random Sobol-like uniform samples.
      2. Select the top-n_restarts candidates as L-BFGS-B starting points.
      3. Return the argmax across all local optima found.
    """
    bounds = [(DOMAIN_LOWER, DOMAIN_UPPER)] * d

    # Warm-start: evaluate EI on random grid
    X_cand    = np.random.uniform(DOMAIN_LOWER, DOMAIN_UPPER,
                                  size=(n_candidates, d))
    ei_cand   = np.array([-expected_improvement(x, gp, scaler, y_best)
                           for x in X_cand])
    top_idx   = np.argsort(ei_cand)[:n_restarts]
    x0_pool   = X_cand[top_idx]

    best_x, best_ei = None, np.inf
    for x0 in x0_pool:
        res = minimize(
            expected_improvement, x0,
            args=(gp, scaler, y_best),
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 500, 'ftol': 1e-12}
        )
        if res.fun < best_ei:
            best_ei = res.fun
            best_x  = res.x

    return best_x, -best_ei   # return (x*, EI value)


def gp_posterior_summary(gp, scaler, X_raw):
    """Return μ and σ at training points (scaled) for diagnostics."""
    X_sc      = scaler.transform(X_raw)
    mu, sigma = gp.predict(X_sc, return_std=True)
    return mu, sigma


print('GP-BO pipeline functions defined.')
print('  build_kernel()          — Matern-5/2 ARD + WhiteKernel')
print('  fit_gp()                — StandardScaler + GPR (10 HP restarts)')
print('  expected_improvement()  — EI (maximisation, ξ=0.01)')
print('  optimise_acquisition()  — Multi-start L-BFGS-B (35 restarts)')
print('  gp_posterior_summary()  — Diagnostic posterior at training points')

GP-BO pipeline functions defined.
  build_kernel()          — Matern-5/2 ARD + WhiteKernel
  fit_gp()                — StandardScaler + GPR (10 HP restarts)
  expected_improvement()  — EI (maximisation, ξ=0.01)
  optimise_acquisition()  — Multi-start L-BFGS-B (35 restarts)
  gp_posterior_summary()  — Diagnostic posterior at training points


In [4]:
# ============================================================
# CELL 4 — RUN GP-BO FOR ALL 8 FUNCTIONS
# ============================================================

gp_suggestions = []   # pure GP-EI suggestions
gp_ei_values   = []

print(f'{'Func':>6}  {'d':>3}  {'y_best':>10}  {'GP-EI x*':>45}  {'EI':>10}')
print('=' * 85)

for fi in range(N_FUNCTIONS):
    X      = func_X[fi]
    y      = func_y[fi]
    d      = dims[fi]
    y_best = y.max()

    # Fit GP
    gp, scaler = fit_gp(X, y)

    # Optimise EI
    x_next, ei_val = optimise_acquisition(gp, scaler, y_best, d)

    gp_suggestions.append(x_next)
    gp_ei_values.append(ei_val)

    x_str = '  '.join([f'{v:.6f}' for v in x_next])
    print(f'  f{fi+1:<2}  {d:>3}  {y_best:>10.4f}  [{x_str}]  {ei_val:>10.6f}')

print('\nGP-EI optimisation complete for all 8 functions.')

  Func    d      y_best                                       GP-EI x*          EI
  f1     2      0.0000  [0.374540  0.950713]    0.000000
  f2     2      0.7237  [0.686429  0.401437]    0.000913
  f3     3     -0.0353  [0.994071  0.048512  0.696666]    0.000000
  f4     4      0.4636  [0.963592  0.058411  0.345779  0.476553]    0.000000
  f5     4   4440.4830  [0.571331  0.415782  0.834512  0.738386]  123.006463
  f6     5     -0.5508  [0.387532  0.480849  0.528589  0.607157  0.649970]    0.000000
  f7     6      2.2502  [0.000000  0.241993  0.488278  0.400543  0.317212  0.681824]    0.369783
  f8     8      9.8596  [0.054311  0.031616  0.889194  0.003175  0.999989  0.369888  0.000000  0.273193]    0.092096

GP-EI optimisation complete for all 8 functions.


  f2     2      0.7237  [0.686429  0.401437]    0.000913


  f3     3     -0.0353  [0.994071  0.048512  0.696666]    0.000000


  f4     4      0.4636  [0.963592  0.058411  0.345779  0.476553]    0.000000


  f5     4   4440.4830  [0.571202  0.415727  0.834507  0.738386]  123.005254


  f6     5     -0.5508  [0.387532  0.480849  0.528589  0.607157  0.649970]    0.000000


  f7     6      2.2502  [0.000000  0.241992  0.488277  0.400542  0.317211  0.681825]    0.369783


  f8     8      9.8596  [0.054318  0.031621  0.889204  0.003178  0.999996  0.369890  0.000000  0.273356]    0.092096

GP-EI optimisation complete for all 8 functions.


In [5]:
# ============================================================
# CELL 5 — PER-FUNCTION STRATEGIC ANALYSIS
# ============================================================
# Combine GP-EI output with expert directional analysis.
# For functions with clear monotone signals or confirmed plateaux,
# the strategic override is applied and documented.

print('=' * 70)
print('PER-FUNCTION ANALYSIS — ROUNDS 1-7 HISTORY')
print('=' * 70)

# ---- f1 --------------------------------------------------------
print()
print('f1 (d=2) — Flat / Near-Zero Landscape')
print('  All 7 outputs ≈ 0. No exploitable gradient signal.')
print('  Best: R7  [0.475, 0.503] → 1.31e-7 (tiny positive)')
print('  Trend R4→R5→R7: values increasing toward [0.475–0.477, 0.501–0.503]')
print('  Strategy: exploit the leading positive cluster with small nudge.')

# ---- f2 --------------------------------------------------------
print()
print('f2 (d=2) — Stochastic / Sharp Local Structure')
print('  R1 & R3 are near-identical inputs yet outputs differ by 0.20.')
print('  Strong evidence of noise or extremely sharp peak.')
print('  Best: R1  [0.6952, 0.3960] → 0.7237')
print('  Strategy: probe x2 slightly below 0.396 to test gradient direction.')

# ---- f3 --------------------------------------------------------
print()
print('f3 (d=3) — Monotone Convergence Toward 0')
f3_hist = [(func_X[2][i], func_y[2][i]) for i in [4, 5, 6]]
for i, (x, y_) in enumerate(f3_hist, 5):
    print(f'  R{i}: [{x[0]:.3f}, {x[1]:.3f}, {x[2]:.3f}] → {y_:.4f}')
delta_f3 = [f3_hist[i][1] - f3_hist[i-1][1] for i in range(1, 3)]
print(f'  Δ per round: {delta_f3[0]:.4f}, {delta_f3[1]:.4f}  (stable ~+0.018)')
print('  Strategy: continue directional extrapolation (x1↓, x3↑).')

# ---- f4 --------------------------------------------------------
print()
print('f4 (d=4) — Local Peak at R6; R7 Overshot')
for i in [2, 5, 6]:
    x = func_X[3][i]; y_ = func_y[3][i]
    print(f'  R{i+1}: [{x[0]:.3f},{x[1]:.3f},{x[2]:.3f},{x[3]:.3f}] → {y_:.4f}')
print('  R6 best. R7 continued the R3→R6 direction but output declined.')
print('  Strategy: fine exploitation between R6 and R3.')

# ---- f5 --------------------------------------------------------
print()
print('f5 (d=4) — Global Maximum Confirmed')
for i in [4, 5, 6]:
    x = func_X[4][i]; y_ = func_y[4][i]
    print(f'  R{i+1}: [{x[0]:.4f},{x[1]:.4f},{x[2]:.4f},{x[3]:.4f}] → {y_:.2f}')
print('  Plateau at 4440.48 confirmed across 3 consecutive rounds.')
print('  Strategy: resubmit confirmed maximum — no exploration benefit.')

# ---- f6 --------------------------------------------------------
print()
print('f6 (d=5) — R1 Holds as Best; Fine Probe Required')
for i in [0, 5, 6]:
    x = func_X[5][i]; y_ = func_y[5][i]
    print(f'  R{i+1}: [{x[0]:.3f},{x[1]:.3f},{x[2]:.3f},{x[3]:.3f},{x[4]:.3f}] → {y_:.4f}')
print('  x4=0.999999, x5=0.000 are confirmed structural constraints.')
print('  Strategy: fine probe x1 slightly below R1 with x3≈0.575.')

# ---- f7 --------------------------------------------------------
print()
print('f7 (d=6) — Strongest Signal: Uniform -0.003 Step Per Round')
for i in [0, 5, 6]:
    x = func_X[6][i]; y_ = func_y[6][i]
    coords = ', '.join([f'{v:.3f}' for v in x])
    print(f'  R{i+1}: [{coords}] → {y_:.4f}')
print('  All active dims decrease by exactly 0.003/round. x1 locked = 0.')
print('  Output improvement decelerating: +0.031, +0.012 → ~+0.005 expected.')
print('  Strategy: continue -0.003 step in all active dimensions.')

# ---- f8 --------------------------------------------------------
print()
print('f8 (d=8) — Near-Plateau; x3 Monotone Positive Gradient')
for i in [2, 0, 5, 6]:
    x = func_X[7][i]; y_ = func_y[7][i]
    print(f'  R{i+1}: x3={x[2]:.6f} → {y_:.7f}')
print('  Only x3 shows a clear gradient. All other dims stable.')
print('  Strategy: increment x3 by +0.002 (0.124→0.126).')

PER-FUNCTION ANALYSIS — ROUNDS 1-7 HISTORY

f1 (d=2) — Flat / Near-Zero Landscape
  All 7 outputs ≈ 0. No exploitable gradient signal.
  Best: R7  [0.475, 0.503] → 1.31e-7 (tiny positive)
  Trend R4→R5→R7: values increasing toward [0.475–0.477, 0.501–0.503]
  Strategy: exploit the leading positive cluster with small nudge.

f2 (d=2) — Stochastic / Sharp Local Structure
  R1 & R3 are near-identical inputs yet outputs differ by 0.20.
  Strong evidence of noise or extremely sharp peak.
  Best: R1  [0.6952, 0.3960] → 0.7237
  Strategy: probe x2 slightly below 0.396 to test gradient direction.

f3 (d=3) — Monotone Convergence Toward 0
  R5: [0.511, 0.215, 0.371] → -0.0707
  R6: [0.490, 0.230, 0.395] → -0.0529
  R7: [0.478, 0.223, 0.408] → -0.0353
  Δ per round: 0.0178, 0.0177  (stable ~+0.018)
  Strategy: continue directional extrapolation (x1↓, x3↑).

f4 (d=4) — Local Peak at R6; R7 Overshot
  R3: [0.440,0.425,0.378,0.397] → 0.2748
  R6: [0.430,0.430,0.375,0.400] → 0.4636
  R7: [0.420,0.44

In [6]:
# ============================================================
# CELL 6 — ROUND 8 FINAL QUERIES (STRATEGIC + GP-EI INFORMED)
# ============================================================
# Final queries integrate GP-EI output with directional analysis.
# Where GP-EI agrees with strategic direction → follow GP-EI.
# Where strategic signal is stronger (plateau/monotone) → override.

round8_queries = [
    # f1: GP-EI near R7 cluster; nudge x1 up, x2 down
    np.array([0.477000, 0.501000]),

    # f2: Re-exploit best region; probe x2=0.394 (below R1's 0.396)
    np.array([0.695000, 0.394000]),

    # f3: Continue directional extrapolation (x1-0.013, x3+0.013 from R7)
    np.array([0.465000, 0.222000, 0.421000]),

    # f4: Fine exploitation between R6 and R3 (best is R6)
    np.array([0.428000, 0.432000, 0.374000, 0.401000]),

    # f5: Confirmed global maximum — hold position
    np.array([0.000000, 0.999999, 0.999999, 0.999999]),

    # f6: Fine probe — x1 slightly below R1 (0.464→0.460), x3 stable at 0.575
    np.array([0.460000, 0.242000, 0.575000, 0.999999, 0.000000]),

    # f7: Continue uniform -0.003 step from R7
    np.array([0.000000, 0.232000, 0.319000, 0.209000, 0.364000, 0.737000]),

    # f8: Increment x3 by +0.002 (monotone positive gradient confirmed)
    np.array([0.064016, 0.008062, 0.126000, 0.000000, 0.999999, 0.381742, 0.031402, 0.806010])
]

# Validation
for fi, x in enumerate(round8_queries):
    assert x.shape[0] == dims[fi], f'Dimension mismatch for f{fi+1}'
    assert np.all(x >= DOMAIN_LOWER) and np.all(x <= DOMAIN_UPPER), \
        f'Bounds violation for f{fi+1}: {x}'

print('All dimension and bounds checks passed.\n')

# Compare GP-EI suggestion vs strategic final
print(f'{'Func':>5}  {'d':>3}  {'GP-EI Suggestion':>42}  {'Strategic Final':>42}')
print('-' * 100)
for fi in range(N_FUNCTIONS):
    gp_str  = '-'.join([f'{v:.6f}' for v in gp_suggestions[fi]])
    fin_str = '-'.join([f'{v:.6f}' for v in round8_queries[fi]])
    print(f'  f{fi+1:<2}  {dims[fi]:>3}  {gp_str:>42}  {fin_str:>42}')

All dimension and bounds checks passed.

 Func    d                            GP-EI Suggestion                             Strategic Final
----------------------------------------------------------------------------------------------------
  f1     2                           0.374540-0.950713                           0.477000-0.501000
  f2     2                           0.686429-0.401437                           0.695000-0.394000
  f3     3                  0.994071-0.048512-0.696666                  0.465000-0.222000-0.421000
  f4     4         0.963592-0.058411-0.345779-0.476553         0.428000-0.432000-0.374000-0.401000
  f5     4         0.571331-0.415782-0.834512-0.738386         0.000000-0.999999-0.999999-0.999999
  f6     5  0.387532-0.480849-0.528589-0.607157-0.649970  0.460000-0.242000-0.575000-0.999999-0.000000
  f7     6  0.000000-0.241993-0.488278-0.400543-0.317212-0.681824  0.000000-0.232000-0.319000-0.209000-0.364000-0.737000
  f8     8  0.054311-0.031616-0.889194-0

In [7]:
# ============================================================
# CELL 7 — GP POSTERIOR DIAGNOSTICS AT ROUND 8 QUERIES
# ============================================================
# Predict μ and σ at proposed query points to quantify
# expected outcome and residual uncertainty.

print('GP Posterior at Round 8 Query Points')
print('=' * 72)
print(f'{'Func':>5}  {'y_best (obs)':>14}  {'μ (pred)':>12}  '
      f'{'σ (pred)':>12}  {'EI':>10}  {'Mode':>12}')
print('-' * 72)

mode_labels = [
    'Exploit-nudge',   # f1
    'Exploit-probe',   # f2
    'Directional',     # f3
    'Fine-exploit',    # f4
    'Hold-max',        # f5
    'Fine-exploit',    # f6
    'Directional',     # f7
    'x3 gradient',     # f8
]

for fi in range(N_FUNCTIONS):
    X      = func_X[fi]
    y      = func_y[fi]
    y_best = y.max()
    gp, scaler = fit_gp(X, y)

    x_q    = round8_queries[fi]
    x_sc   = scaler.transform(x_q.reshape(1, -1))
    mu, sg = gp.predict(x_sc, return_std=True)
    mu, sg = float(mu[0]), float(sg[0])

    ei_val = -expected_improvement(x_q, gp, scaler, y_best)

    print(f'  f{fi+1:<2}  {y_best:>14.6f}  {mu:>12.6f}  '
          f'{sg:>12.6f}  {ei_val:>10.6f}  {mode_labels[fi]:>12}')

print()
print('Note: High σ at query point = exploration value.')
print('      Low σ, high μ         = pure exploitation.')

GP Posterior at Round 8 Query Points
 Func    y_best (obs)      μ (pred)      σ (pred)          EI          Mode
------------------------------------------------------------------------
  f1         0.000000      0.000000      0.000000    0.000000  Exploit-nudge
  f2         0.723740      0.528056      0.032649    0.000000  Exploit-probe
  f3        -0.035298     -0.019700      0.003575    0.005688   Directional
  f4         0.463617      0.466031      0.007710    0.000662  Fine-exploit
  f5      4440.482960   4440.480858      0.192970    0.071084      Hold-max
  f6        -0.550775     -0.591326      0.049662    0.003999  Fine-exploit
  f7         2.250193      2.260488      0.001319    0.000687   Directional
  f8         9.859577      9.859639      0.000208    0.000000   x3 gradient

Note: High σ at query point = exploration value.
      Low σ, high μ         = pure exploitation.


  f2         0.723740      0.528056      0.032649    0.000000  Exploit-probe


  f3        -0.035298     -0.019700      0.003575    0.005688   Directional


  f4         0.463617      0.466031      0.007710    0.000662  Fine-exploit


  f5      4440.482960   4440.480858      0.192970    0.071084      Hold-max


  f6        -0.550775     -0.591326      0.049662    0.003999  Fine-exploit


  f7         2.250193      2.260488      0.001319    0.000687   Directional


  f8         9.859577      9.859639      0.000208    0.000000   x3 gradient

Note: High σ at query point = exploration value.
      Low σ, high μ         = pure exploitation.


In [8]:
# ============================================================
# CELL 8 — ROUND 8 SUBMISSION STRINGS (PORTAL FORMAT)
# ============================================================
# Format: x1-x2-...-xn
# Each xi: begins with 0, six decimal places.

print('=' * 70)
print('ROUND 8 — FINAL SUBMISSION STRINGS (W19 / Module 19.1)')
print('=' * 70)
print()

func_labels = [
    'F1 (d=2)', 'F2 (d=2)', 'F3 (d=3)', 'F4 (d=4)',
    'F5 (d=4)', 'F6 (d=5)', 'F7 (d=6)', 'F8 (d=8)'
]

strategies = [
    'Exploit near R7 positive cluster',
    'Re-exploit best region; x2 probe below 0.396',
    'Directional continuation: x1↓ x3↑',
    'Fine exploitation near R6 peak',
    'Confirmed global maximum — hold',
    'Fine probe: x1 ↓ from R1 best',
    'Uniform -0.003 step from R7',
    'x3 monotone gradient: 0.124 → 0.126',
]

submission_strings = []
for fi in range(N_FUNCTIONS):
    s = '-'.join([f'{v:.6f}' for v in round8_queries[fi]])
    submission_strings.append(s)
    print(f'{func_labels[fi]}:  {s}')
    print(f'    Strategy: {strategies[fi]}')
    print(f'    y_best (Rounds 1-7): {func_y[fi].max():.6f}')
    print()

# One-line copy block
print()
print('--- COPY-PASTE BLOCK ---')
for fi in range(N_FUNCTIONS):
    print(submission_strings[fi])

ROUND 8 — FINAL SUBMISSION STRINGS (W19 / Module 19.1)

F1 (d=2):  0.477000-0.501000
    Strategy: Exploit near R7 positive cluster
    y_best (Rounds 1-7): 0.000000

F2 (d=2):  0.695000-0.394000
    Strategy: Re-exploit best region; x2 probe below 0.396
    y_best (Rounds 1-7): 0.723740

F3 (d=3):  0.465000-0.222000-0.421000
    Strategy: Directional continuation: x1↓ x3↑
    y_best (Rounds 1-7): -0.035298

F4 (d=4):  0.428000-0.432000-0.374000-0.401000
    Strategy: Fine exploitation near R6 peak
    y_best (Rounds 1-7): 0.463617

F5 (d=4):  0.000000-0.999999-0.999999-0.999999
    Strategy: Confirmed global maximum — hold
    y_best (Rounds 1-7): 4440.482960

F6 (d=5):  0.460000-0.242000-0.575000-0.999999-0.000000
    Strategy: Fine probe: x1 ↓ from R1 best
    y_best (Rounds 1-7): -0.550775

F7 (d=6):  0.000000-0.232000-0.319000-0.209000-0.364000-0.737000
    Strategy: Uniform -0.003 step from R7
    y_best (Rounds 1-7): 2.250193

F8 (d=8):  0.064016-0.008062-0.126000-0.000000-0.999

In [9]:
# ============================================================
# CELL 9 — CUMULATIVE BEST TRACKER (ALL ROUNDS)
# ============================================================

print('Cumulative Best Values by Round')
print('=' * 90)
header = f'{'Round':>7}' + ''.join([f'  {"f"+str(i+1):>12}' for i in range(N_FUNCTIONS)])
print(header)
print('-' * 90)

best_so_far = [-np.inf] * N_FUNCTIONS

for rnd in range(N_ROUNDS):
    row = f'  R{rnd+1:>4}'
    for fi in range(N_FUNCTIONS):
        y_val = all_outputs[rnd][fi]
        if y_val > best_so_far[fi]:
            best_so_far[fi] = y_val
        row += f'  {best_so_far[fi]:>12.4f}'
    print(row)

print('-' * 90)
# Show final best
final_row = '  Final'
for fi in range(N_FUNCTIONS):
    final_row += f'  {best_so_far[fi]:>12.4f}'
print(final_row)

print()
print('Key observations:')
print('  f1 — Effectively 0 throughout; tiny positive peak ~1.3e-7 at R7')
print('  f2 — Best at R1 (0.7237); high variance across rounds')
print('  f3 — Monotone improvement from -0.089 to -0.035 (R1→R7)')
print('  f4 — Peaked at R6 (0.4636); R7 regression confirms local max')
print('  f5 — Plateau at 4440.48 confirmed R5/R6/R7')
print('  f6 — R1 still best (-0.5508); function resists improvement')
print('  f7 — Steady gain: 2.207 → 2.250 across R1→R7')
print('  f8 — Near-plateau at 9.8596; x3 the only active gradient')

Cumulative Best Values by Round
  Round            f1            f2            f3            f4            f5            f6            f7            f8
------------------------------------------------------------------------------------------
  R   1       -0.0000        0.7237       -0.0891        0.2596     2105.9282       -0.5508        2.2073        9.8595
  R   2        0.0000        0.7237       -0.0891        0.2596     2105.9282       -0.5508        2.2073        9.8595
  R   3        0.0000        0.7237       -0.0891        0.2748     2932.6950       -0.5508        2.2073        9.8595
  R   4        0.0000        0.7237       -0.0891        0.2748     2932.6950       -0.5508        2.2073        9.8595
  R   5        0.0000        0.7237       -0.0707        0.2748     4440.4809       -0.5508        2.2073        9.8595
  R   6        0.0000        0.7237       -0.0529        0.4636     4440.4830       -0.5508        2.2378        9.8595
  R   7        0.0000        0.7237  

---

## Pipeline Notes

### Kernel Choice
**Matérn-5/2 with ARD** is the standard for black-box functions of unknown smoothness. ARD (Automatic Relevance Determination) assigns a separate length-scale per dimension, allowing the GP to identify which inputs are most influential — critical in higher-dimensional functions (f6, f7, f8) where some dimensions dominate.

### Noise Handling
The **WhiteKernel** component models observation noise, which is particularly important for f2 where R1 and R3 — near-identical inputs — produced outputs differing by 0.20. Without a noise kernel the GP would over-interpolate these conflicting observations and produce a poor surrogate.

### EI Formulation
All 8 functions use the **maximisation EI**: `EI(x) = (μ(x) - y_best - ξ) · Φ(Z) + σ · φ(Z)`. The exploration parameter `ξ = 0.01` provides a small constant incentive to explore beyond the current best, preventing premature convergence.

### Acquisition Optimisation
EI is highly multimodal — a single gradient descent from a random start routinely misses the global maximum. **35 restarts** with a warm-start from the top-35 of 5,000 random evaluations provides robust coverage of the acquisition landscape.

### LLM Reflection (Module 19.1 Prompt)
With 17 data points (10 initial + 7 submitted rounds), the GP posterior is still sparse relative to the input dimensionality of f8 (d=8). The risk of *prompt overfitting* in this analogy maps to over-exploitation: concentrating all queries near the current best risks missing a global optimum in an unexplored region. For f5 (confirmed plateau) the risk is zero; for f1 (near-zero everywhere) it remains high. The directional signal in f7 is the cleanest case where exploitation is unambiguously justified.

---
*Gian Franco Cattaneo — SALOV S.p.A. / Imperial Business School*